# 01. Сбор данных с РешуЕГЭ (sdamgia.ru)

**Цель:** собрать тексты заданий ЕГЭ с пронумерованными предложениями, извлечь разметку границ, сохранить в JSONL.

**Источник:** тексты заданий с сайта sdamgia.ru, где предложения пронумерованы: `(1)Текст первого предложения. (2)Текст второго...`

**Результат:** файл `data/processed/sentences.jsonl` с полями: `id, source, raw_text, clean_text, sentences, num_sentences`

In [1]:
import requests
import time
import json
import re
import hashlib
from bs4 import BeautifulSoup
from pathlib import Path

In [2]:
BASE_URL = "https://rus-ege.sdamgia.ru"
CATALOG_URL = BASE_URL + "/test?theme={theme}&page={page}"
PROBLEM_URL = BASE_URL + "/problem?id={task_id}"

THEMES = [399]  # задание 27 — сочинение

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DELAY = 1.5
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
}

## 2. Сбор ID заданий из каталога

Каталог задания 27 на sdamgia.ru: тема 399. Страницы пагинируются через `&page=N`. На каждой странице ~7-13 заданий, всего ~130-150. Парсим ссылки вида `href="/problem?id=XXXXX"`.

In [3]:
def collect_task_ids(theme: int, max_pages: int = 30) -> list[int]:
    """Собираем ID заданий из каталога sdamgia.ru по теме."""
    all_ids = []
    empty_streak = 0

    for page in range(1, max_pages + 1):
        url = CATALOG_URL.format(theme=theme, page=page)
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"  Страница {page}: ошибка — {e}")
            empty_streak += 1
            if empty_streak >= 2:
                break
            time.sleep(DELAY)
            continue

        soup = BeautifulSoup(resp.text, "lxml")
        links = soup.select('a[href*="/problem?id="]')
        ids_on_page = set()
        for a in links:
            m = re.search(r'id=(\d+)', a.get("href", ""))
            if m:
                ids_on_page.add(int(m.group(1)))

        if not ids_on_page:
            empty_streak += 1
            print(f"  Страница {page}: пусто ({empty_streak}/2)")
            if empty_streak >= 2:
                break
        else:
            empty_streak = 0
            all_ids.extend(ids_on_page)
            print(f"  Страница {page}: найдено {len(ids_on_page)} ID")

        time.sleep(DELAY)

    if page == max_pages:
        print(f"  WARN: достигнут лимит {max_pages} страниц")

    return sorted(set(all_ids))


task_ids = []
for theme in THEMES:
    print(f"Тема {theme}:")
    ids = collect_task_ids(theme)
    task_ids.extend(ids)
    print(f"  Итого: {len(ids)} заданий\n")

task_ids = sorted(set(task_ids))
print(f"Всего уникальных ID: {len(task_ids)}")
print(f"Первые 10: {task_ids[:10]}")

if not task_ids:
    raise RuntimeError("Не удалось собрать ни одного ID задания — проверьте сеть и THEMES")

Тема 399:


  Страница 1: найдено 35 ID


  Страница 2: найдено 35 ID


  Страница 3: найдено 35 ID


  Страница 4: найдено 35 ID


...

  Страница 17: найдено 23 ID


  Страница 18: найдено 18 ID


  Страница 19: найдено 16 ID


  Страница 20: пусто (1/2)


  Страница 21: пусто (2/2)
  Итого: 716 заданий

Всего уникальных ID: 716
Первые 10: [2758, 2759, 2760, 2761, 2762, 2763, 2764, 2765, 2766, 2767]


## 3. Демо: парсинг одной страницы

Прежде чем скачивать все задания, разберём HTML-структуру одного. Текст автора находится в `div.probtext` → `p.left_margin`. После текста идёт атрибуция автора `(По ...)` и биографическая сноска — их нужно исключить.

In [4]:
# Скачаем одну страницу для изучения
assert task_ids, "Нет ID заданий — невозможно запустить демо"

demo_id = task_ids[0]
demo_url = PROBLEM_URL.format(task_id=demo_id)
print(f"Скачиваем задание {demo_id}: {demo_url}\n")

demo_resp = requests.get(demo_url, headers=HEADERS, timeout=30)
demo_resp.raise_for_status()

demo_soup = BeautifulSoup(demo_resp.text, "lxml")

probtext = demo_soup.select_one("div.probtext")
if probtext:
    print(f"data-text_id = {probtext.get('data-text_id')}")
    print(f"\nПараграфы (p.left_margin):")
    for i, p in enumerate(probtext.select("p.left_margin")):
        text = p.get_text(strip=True)
        print(f"  [{i}] {text[:100]}{'...' if len(text) > 100 else ''}")
else:
    raise ValueError(f"div.probtext не найден для задания {demo_id} — выберите другой demo_id")

Скачиваем задание 2758: https://rus-ege.sdamgia.ru/problem?id=2758



data-text_id = 180

Параграфы (p.left_margin):
  [0] 
  [1] (1)Это было не­сколь­ко лет тому назад. (2)Все, со­би­ра­ясь празд­но­вать Рож­де­ство, го­то­ви­ли ...
  [2] (6)А я, бес­при­ют­ный ски­та­лец, был оди­нок в чужой стра­не  — ни семьи, ни друга, и мне ка­за­ло...
  [3] (11)«До­ро­гое дитя моё, Ни­ко­лень­ка! (12)Ты жа­лу­ешь­ся мне на своё оди­но­че­ство, и если бы ты...
  [4] (17)Ви­дишь ли, сынок, че­ло­век оди­нок тогда, когда он ни­ко­го не любит. (18)А если любит, то ему...
  [5] (21)Я уже слышу твоё воз­ра­же­ние, что сча­стье не в том толь­ко, чтобы лю­бить, но и в том, чтобы ...
  [6] (31)Я до­чи­ты­вал ма­ми­но пись­мо со сле­за­ми на гла­зах. (32)Из дали про­шед­ших лет я снова усл...
  [7] (33)И тогда я по­ду­мал, что наша лю­бовь  — это нить, ко­то­рой мы при­вя­за­ны к лю­би­мо­му че­ло...


In [5]:
# Демо: извлечение текста и парсинг предложений
assert probtext is not None, "probtext не найден — ячейка выше должна была вызвать ошибку"

paragraphs = probtext.find_all("p")
lines = []
for p in paragraphs:
    first_child = next(p.children, None)
    if first_child and getattr(first_child, "name", None) == "sup":
        sup_text = first_child.get_text().strip()
        if sup_text.isdigit():
            continue
    for sup in p.select("sup"):
        sup.decompose()
    text = p.get_text()
    text = text.replace("\u00ad", "")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u2061", "")
    text = text.replace("\u2060", "")
    text = text.replace("\u202f", " ")
    text = text.replace("\t", " ")
    text = text.strip()
    if not text:
        continue
    if re.match(r'^\([Пп]о\s', text) and not re.match(r'^\(\d+\)', text):
        break
    if text.startswith("Источник текста"):
        continue
    if text.startswith('*') and not re.match(r'^\(\d+\)', text):
        continue
    if re.match(r'^\d+[А-ЯЁа-яё]', text) and not re.match(r'^\(\d+\)', text):
        continue
    if re.match(r'^[А-ЯЁ][а-яё]+ [А-ЯЁ]', text) and re.search(r'\(\d{4}[−–\-]\d{4}\)', text):
        continue
    lines.append(text)

raw_text = " ".join(lines)

# Post-process: inline-метаданные из того же <p>
raw_text = re.sub(r'\s*\([Пп]о\s+(?:по\s+)?[^()]*[А-ЯЁ]\.[^()]*\)\s*$', '', raw_text)
raw_text = re.sub(r'\s*\([Пп]о\s+[^()]+\*\s*\)\s*$', '', raw_text)
raw_text = re.sub(r'\s*\*[А-ЯЁ][а-яё]+\s[А-ЯЁ].*?\(\d{4}[−–\-]\d{4}\).*$', '', raw_text)
raw_text = re.sub(r'(?<=[.!?…»)"])\s+\d+[А-ЯЁ][а-яё]+\s*[—–\-]\s.*$', '', raw_text)
raw_text = re.sub(r'\s*Источник текста:.*$', '', raw_text)
raw_text = raw_text.strip()

print(f"raw_text ({len(raw_text)} символов):")
print(raw_text[:500])
print("..." if len(raw_text) > 500 else "")

# Парсим предложения по маркерам (N) — числа < 1000
all_markers = list(re.finditer(r'\((\d+)\)', raw_text))
markers = [m for m in all_markers if int(m.group(1)) < 1000]

clean_text = raw_text
for m in reversed(markers):
    clean_text = clean_text[:m.start()] + clean_text[m.end():]
clean_text = re.sub(r'  +', ' ', clean_text).strip()

sentences = []
min_pos = 0
for i, m in enumerate(markers):
    idx = int(m.group(1))
    start_raw = m.end()
    end_raw = markers[i + 1].start() if i + 1 < len(markers) else len(raw_text)
    sent_text = raw_text[start_raw:end_raw].strip()
    if not sent_text:
        continue
    # Нормализуем пробелы так же, как в clean_text
    sent_text = re.sub(r'  +', ' ', sent_text)
    found = clean_text.find(sent_text, min_pos)
    if found == -1:
        print(f"  ОШИБКА: предложение {idx} не найдено в clean_text")
        continue
    start_clean = found
    end_clean = found + len(sent_text)
    min_pos = end_clean
    sentences.append({"idx": idx, "start": start_clean, "end": end_clean, "text": sent_text})

print(f"\nНайдено {len(sentences)} предложений:")
for s in sentences[:5]:
    print(f"  ({s['idx']}) [{s['start']}:{s['end']}] {s['text'][:80]}...")
if len(sentences) > 5:
    print(f"  ... и ещё {len(sentences) - 5}")

errors = 0
for s in sentences:
    actual = clean_text[s["start"]:s["end"]]
    if actual != s["text"]:
        print(f"  ОШИБКА в предложении {s['idx']}: ожидалось '{s['text'][:50]}', получено '{actual[:50]}'")
        errors += 1
print(f"\nПроверка позиций: {'OK' if errors == 0 else f'{errors} ошибок'}")

raw_text (2813 символов):
(1)Это было несколько лет тому назад. (2)Все, собираясь праздновать Рождество, готовили ёлки и подарки. (3)Витрины магазинов и окна домов сияли праздничными огнями. (4)Повсюду были развешаны гирлянды и большущие цветные шары. (5)Толпа горожан, пёстрая, шумная и весёлая, заполняла улицы. (6)А я, бесприютный скиталец, был одинок в чужой стране  — ни семьи, ни друга, и мне казалось, что я покинут и забыт всеми. (7)Вокруг была только пустота и не было любви: дальний город, чужие люди, холодные сердц
...

Найдено 35 предложений:
  (1) [0:34] Это было несколько лет тому назад....
  (2) [35:97] Все, собираясь праздновать Рождество, готовили ёлки и подарки....
  (3) [98:155] Витрины магазинов и окна домов сияли праздничными огнями....
  (4) [156:213] Повсюду были развешаны гирлянды и большущие цветные шары....
  (5) [214:272] Толпа горожан, пёстрая, шумная и весёлая, заполняла улицы....
  ... и ещё 30

Проверка позиций: OK


## 4. Функции парсинга

- `extract_raw_text(html)` — из HTML извлекает сырой текст с нумерацией, убирает автора и сноску
- `parse_sentences(raw_text)` — из текста с `(1)...(2)...` извлекает список предложений и clean_text

**CSS-селекторы:** `div.probtext` → все `<p>` (включая без класса — диалоги, продолжения)

**Очистка текста:** `\u00ad`, `\u00a0`, `\u2061`, `\u2060`, `\u202f`, `\t`, `<sup>` decompose

**Фильтрация параграфов:** `(По/по ...)` break, `Источник текста` skip, `*bio` skip, `1Def — ...` skip, `Name (YYYY-YYYY)` skip

**Post-processing raw_text (inline-метаданные из того же `<p>`):**
- Атрибуция с инициалом/астериском, биография, сноски-определения, `Источник текста`

In [6]:
def extract_raw_text(html_content: str) -> tuple[str | None, str | None]:
    """Извлекает сырой текст с нумерацией из HTML страницы задания.

    Returns:
        (raw_text, data_text_id) или (None, None) если текст не найден.
    """
    soup = BeautifulSoup(html_content, "lxml")
    probtext = soup.select_one("div.probtext")

    if not probtext:
        return None, None

    data_text_id = probtext.get("data-text_id")

    paragraphs = probtext.find_all("p")
    lines = []
    for p in paragraphs:
        # Сноска-определение с <sup>: <sup>1</sup>Архивариус — ...
        first_child = next(p.children, None)
        if first_child and getattr(first_child, "name", None) == "sup":
            sup_text = first_child.get_text().strip()
            if sup_text.isdigit():
                continue

        # Удаляем <sup> (inline сноски вида мола¹) перед извлечением текста
        for sup in p.select("sup"):
            sup.decompose()

        text = p.get_text()
        text = text.replace("\u00ad", "")   # мягкие переносы
        text = text.replace("\u00a0", " ")  # non-breaking space
        text = text.replace("\u2061", "")   # function application (invisible)
        text = text.replace("\u2060", "")   # word joiner
        text = text.replace("\u202f", " ")  # narrow no-break space → пробел
        text = text.replace("\t", " ")      # табуляция → пробел
        text = text.strip()
        if not text:
            continue
        if re.match(r'^\([Пп]о\s', text) and not re.match(r'^\(\d+\)', text):
            break
        if text.startswith("Источник текста"):
            continue
        if text.startswith('*') and not re.match(r'^\(\d+\)', text):
            continue
        if re.match(r'^\d+[А-ЯЁа-яё]', text) and not re.match(r'^\(\d+\)', text):
            continue
        if re.match(r'^[А-ЯЁ][а-яё]+ [А-ЯЁ]', text) and re.search(r'\(\d{4}[−–\-]\d{4}\)', text):
            continue
        lines.append(text)

    if not lines:
        return None, data_text_id

    raw_text = " ".join(lines)

    # Post-process: inline-метаданные из того же <p> что и нумерованный текст
    raw_text = re.sub(r'\s*\([Пп]о\s+(?:по\s+)?[^()]*[А-ЯЁ]\.[^()]*\)\s*$', '', raw_text)
    raw_text = re.sub(r'\s*\([Пп]о\s+[^()]+\*\s*\)\s*$', '', raw_text)
    raw_text = re.sub(r'\s*\*[А-ЯЁ][а-яё]+\s[А-ЯЁ].*?\(\d{4}[−–\-]\d{4}\).*$', '', raw_text)
    raw_text = re.sub(r'(?<=[.!?…»)"])\s+\d+[А-ЯЁ][а-яё]+\s*[—–\-]\s.*$', '', raw_text)
    raw_text = re.sub(r'\s*Источник текста:.*$', '', raw_text)
    raw_text = raw_text.strip()

    if not raw_text:
        return None, data_text_id

    return raw_text, data_text_id

In [7]:
def parse_sentences(raw_text: str) -> tuple[str, list[dict]]:
    """Парсит текст с маркерами (1)...(2)... в список предложений.

    Каждый маркер (N) где N < 1000 считается границей предложения.
    Числа >= 1000 (годы вроде (1924)) отбрасываются.
    """
    if not raw_text:
        return "", []

    all_markers = list(re.finditer(r'\((\d+)\)', raw_text))
    markers = [m for m in all_markers if int(m.group(1)) < 1000]

    if not markers:
        return raw_text, []

    remove_ranges = [(m.start(), m.end()) for m in markers]

    parts = []
    prev_end = 0
    for rs, re_ in remove_ranges:
        parts.append(raw_text[prev_end:rs])
        prev_end = re_
    parts.append(raw_text[prev_end:])
    clean_text = "".join(parts)
    clean_text = re.sub(r'  +', ' ', clean_text).strip()

    sentences = []
    min_pos = 0
    for i, m in enumerate(markers):
        idx = int(m.group(1))
        start_raw = m.end()
        end_raw = markers[i + 1].start() if i + 1 < len(markers) else len(raw_text)

        sent_text = raw_text[start_raw:end_raw].strip()
        if not sent_text:
            continue

        # Нормализуем пробелы так же, как в clean_text
        sent_text = re.sub(r'  +', ' ', sent_text)

        found = clean_text.find(sent_text, min_pos)
        if found == -1:
            print(f"  WARN: предложение {idx} — не найдено в clean_text, пропускаем")
            continue

        start_clean = found
        end_clean = found + len(sent_text)
        min_pos = end_clean

        sentences.append({"idx": idx, "start": start_clean, "end": end_clean, "text": sent_text})

    return clean_text, sentences


# Проверяем на демо-примере
raw, text_id = extract_raw_text(demo_resp.text)
assert raw is not None, f"Не удалось извлечь текст из демо-страницы {demo_id}"

clean, sents = parse_sentences(raw)
print(f"data_text_id: {text_id}")
print(f"Предложений: {len(sents)}")
print(f"clean_text: {clean[:200]}...")

ok = all(clean[s["start"]:s["end"]] == s["text"] for s in sents)
print(f"Позиции корректны: {ok}")

data_text_id: 180
Предложений: 35
clean_text: Это было несколько лет тому назад. Все, собираясь праздновать Рождество, готовили ёлки и подарки. Витрины магазинов и окна домов сияли праздничными огнями. Повсюду были развешаны гирлянды и большущие ...
Позиции корректны: True


## 5. Массовое скачивание HTML

Скачиваем HTML-страницы всех заданий в `data/raw/`. Пауза между запросами — 1.5 сек. Уже скачанные файлы пропускаем (идемпотентность).

In [8]:
def fetch_task(task_id: int, retries: int = 3) -> tuple[str | None, bool]:
    """Скачивает HTML страницы задания. Atomic write + retry."""
    filepath = DATA_RAW / f"{task_id}.html"

    if filepath.exists() and filepath.stat().st_size > 100:
        return filepath.read_text(encoding="utf-8"), True

    url = PROBLEM_URL.format(task_id=task_id)
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            tmp = filepath.with_suffix(".tmp")
            tmp.write_text(resp.text, encoding="utf-8")
            tmp.rename(filepath)
            return resp.text, False
        except requests.RequestException as e:
            if attempt < retries - 1:
                wait = DELAY * (2 ** attempt)
                print(f"  Retry {attempt + 1}/{retries} для {task_id}: {e}, жду {wait:.1f}s")
                time.sleep(wait)
            else:
                print(f"  Ошибка {task_id} после {retries} попыток: {e}")
                return None, False


downloaded, cached, errors = 0, 0, 0
for i, task_id in enumerate(task_ids):
    html, was_cached = fetch_task(task_id)
    if html is None:
        errors += 1
    elif was_cached:
        cached += 1
    else:
        downloaded += 1
        time.sleep(DELAY)

    if (i + 1) % 20 == 0:
        print(f"  Прогресс: {i + 1}/{len(task_ids)} (скачано: {downloaded}, кеш: {cached}, ошибок: {errors})")

print(f"\nГотово! Скачано: {downloaded}, из кеша: {cached}, ошибок: {errors}")
print(f"Файлов в data/raw/: {len(list(DATA_RAW.glob('*.html')))}")

  Прогресс: 20/716 (скачано: 0, кеш: 20, ошибок: 0)
  Прогресс: 40/716 (скачано: 0, кеш: 40, ошибок: 0)
  Прогресс: 60/716 (скачано: 0, кеш: 60, ошибок: 0)


  Прогресс: 80/716 (скачано: 0, кеш: 80, ошибок: 0)
  Прогресс: 100/716 (скачано: 0, кеш: 100, ошибок: 0)
  Прогресс: 120/716 (скачано: 0, кеш: 120, ошибок: 0)
  Прогресс: 140/716 (скачано: 0, кеш: 140, ошибок: 0)


  Прогресс: 160/716 (скачано: 0, кеш: 160, ошибок: 0)


  Прогресс: 180/716 (скачано: 0, кеш: 180, ошибок: 0)
  Прогресс: 200/716 (скачано: 0, кеш: 200, ошибок: 0)


  Прогресс: 220/716 (скачано: 0, кеш: 220, ошибок: 0)
  Прогресс: 240/716 (скачано: 0, кеш: 240, ошибок: 0)
  Прогресс: 260/716 (скачано: 0, кеш: 260, ошибок: 0)


...

  Прогресс: 400/716 (скачано: 0, кеш: 400, ошибок: 0)
  Прогресс: 420/716 (скачано: 0, кеш: 420, ошибок: 0)
  Прогресс: 440/716 (скачано: 0, кеш: 440, ошибок: 0)


  Прогресс: 460/716 (скачано: 0, кеш: 460, ошибок: 0)
  Прогресс: 480/716 (скачано: 0, кеш: 480, ошибок: 0)


  Прогресс: 500/716 (скачано: 0, кеш: 500, ошибок: 0)
  Прогресс: 520/716 (скачано: 0, кеш: 520, ошибок: 0)
  Прогресс: 540/716 (скачано: 0, кеш: 540, ошибок: 0)
  Прогресс: 560/716 (скачано: 0, кеш: 560, ошибок: 0)


  Прогресс: 580/716 (скачано: 0, кеш: 580, ошибок: 0)
  Прогресс: 600/716 (скачано: 0, кеш: 600, ошибок: 0)
  Прогресс: 620/716 (скачано: 0, кеш: 620, ошибок: 0)
  Прогресс: 640/716 (скачано: 0, кеш: 640, ошибок: 0)
  Прогресс: 660/716 (скачано: 0, кеш: 660, ошибок: 0)
  Прогресс: 680/716 (скачано: 0, кеш: 680, ошибок: 0)


  Прогресс: 700/716 (скачано: 0, кеш: 700, ошибок: 0)

Готово! Скачано: 0, из кеша: 716, ошибок: 0
Файлов в data/raw/: 716


## 6. Сборка датасета

Обрабатываем все скачанные HTML → извлекаем текст → парсим предложения → формируем записи JSONL.

Дедупликация по `data_text_id`: разные задания могут ссылаться на один и тот же текст автора.

**Формат записи:**
| Поле | Описание |
|------|----------|
| `id` | `text_{data_text_id}` — уникальный ID текста |
| `source` | `reshuege_{task_id}` — откуда взяли |
| `raw_text` | Текст с нумерацией `(1)...(2)...` |
| `clean_text` | Текст без номеров |
| `sentences` | Список `{idx, start, end, text}` |
| `num_sentences` | Количество предложений |

In [9]:
dataset = []
seen_text_ids = set()
parse_errors = []

html_files = sorted(DATA_RAW.glob("*.html"))
print(f"HTML-файлов для обработки: {len(html_files)}\n")

for filepath in html_files:
    task_id = filepath.stem
    html_content = filepath.read_text(encoding="utf-8")

    raw_text, data_text_id = extract_raw_text(html_content)

    if raw_text is None:
        parse_errors.append((task_id, "no probtext or empty text"))
        continue

    dedup_key = data_text_id if data_text_id else hashlib.md5(raw_text.encode()).hexdigest()
    if dedup_key in seen_text_ids:
        continue
    seen_text_ids.add(dedup_key)

    clean_text, sentences = parse_sentences(raw_text)

    if not sentences:
        parse_errors.append((task_id, "no sentences found"))
        continue

    record = {
        "id": f"text_{data_text_id}" if data_text_id else f"text_hash_{dedup_key[:8]}",
        "source": f"reshuege_{task_id}",
        "raw_text": raw_text,
        "clean_text": clean_text,
        "sentences": sentences,
        "num_sentences": len(sentences)
    }
    dataset.append(record)

print(f"Записей в датасете: {len(dataset)}")
print(f"Уникальных текстов (dedup): {len(seen_text_ids)}")
if parse_errors:
    print(f"\nОшибки парсинга ({len(parse_errors)}):")
    for tid, err in parse_errors[:10]:
        print(f"  task {tid}: {err}")
    if len(parse_errors) > 10:
        print(f"  ... и ещё {len(parse_errors) - 10}")

HTML-файлов для обработки: 716



Записей в датасете: 90
Уникальных текстов (dedup): 90


## 7. Валидация и сохранение

Проверяем корректность данных:
- Позиции `start:end` в `clean_text` совпадают с `text` каждого предложения
- Предложения покрывают текст без пропусков
- Базовая статистика по датасету

In [10]:
# Валидация
if not dataset:
    print("Dataset пуст — пропускаем валидацию и статистику")
else:
    total_errors = 0
    total_sentences = 0

    for record in dataset:
        ct = record["clean_text"]
        for s in record["sentences"]:
            total_sentences += 1
            actual = ct[s["start"]:s["end"]]
            if actual != s["text"]:
                total_errors += 1
                if total_errors <= 3:
                    print(f"ОШИБКА в {record['id']}, предложение {s['idx']}:")
                    print(f"  Ожидалось: {s['text'][:80]}")
                    print(f"  Получено:  {actual[:80]}")

    print(f"Проверено записей: {len(dataset)}")
    print(f"Проверено предложений: {total_sentences}")
    print(f"Ошибок позиций: {total_errors}")

    num_sents = [r["num_sentences"] for r in dataset]
    text_lens = [len(r["clean_text"]) for r in dataset]
    sent_lens = [len(s["text"]) for r in dataset for s in r["sentences"]]

    print(f"\n--- Статистика ---")
    print(f"Текстов: {len(dataset)}")
    print(f"Предложений всего: {sum(num_sents)}")
    print(f"Предложений на текст: min={min(num_sents)}, max={max(num_sents)}, avg={sum(num_sents)/len(num_sents):.1f}")
    print(f"Длина текста (символов): min={min(text_lens)}, max={max(text_lens)}, avg={sum(text_lens)/len(text_lens):.0f}")
    print(f"Длина предложения (символов): min={min(sent_lens)}, max={max(sent_lens)}, avg={sum(sent_lens)/len(sent_lens):.0f}")

Проверено записей: 90
Проверено предложений: 4359
Ошибок позиций: 0

--- Статистика ---
Текстов: 90
Предложений всего: 4359
Предложений на текст: min=18, max=97, avg=48.4
Длина текста (символов): min=2006, max=5476, avg=3664
Длина предложения (символов): min=2, max=395, avg=74


In [11]:
# Сохранение в JSONL (атомарная запись)
output_path = DATA_PROCESSED / "sentences.jsonl"
tmp_path = output_path.with_suffix(".tmp")

if not dataset:
    print("Dataset пуст — нечего сохранять")
else:
    with open(tmp_path, "w", encoding="utf-8") as f:
        for record in dataset:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    tmp_path.rename(output_path)

    print(f"Сохранено в {output_path}")
    print(f"Размер файла: {output_path.stat().st_size / 1024:.1f} КБ")

    with open(output_path, "r", encoding="utf-8") as f:
        loaded = [json.loads(line) for line in f]
    print(f"Проверка чтения: {len(loaded)} записей загружено")

Сохранено в ../data/processed/sentences.jsonl
Размер файла: 1993.9 КБ
Проверка чтения: 90 записей загружено


## Итоги

**Что сделано:**
- Собраны ID заданий из каталога темы 399 (задание 27 ЕГЭ)
- Скачаны HTML-страницы всех заданий
- Извлечены тексты авторов с пронумерованными предложениями
- Дедуплицированы тексты по `data_text_id`
- Данные сохранены в `data/processed/sentences.jsonl`

**Далее:** `02_eda.ipynb` — разведочный анализ данных (распределение длин, визуализация, edge cases)